# M5 weekly demand forecast: baseline-решение

Прогноз недельного спроса item × store на 4 недели вперёд. Автор: Dmitrii Gertsovskii.

# 0. Задача

Прогноз **недельного спроса** на товар × магазин на **4 недели вперёд** на данных M5
(недельные продажи Walmart, открытый датасет Kaggle). Baseline-решение: простые модели как
точка отсчёта плюс одна усложнённая модель с подбором гиперпараметров, оценка совокупной
метрикой и разбор результатов.

Чтобы ноутбук считался быстро и целиком, берём подвыборку категории **FOODS**: 14370 рядов
товар × магазин по 10 магазинам и 3 штатам. Методика переносится на все 30490 рядов M5.

Горизонт 4 недели, совокупная метрика **WRMSSE** (взвешенный RMSSE по уровням иерархии M5).

In [ ]:
#| label: setup
#| code-fold: true
import warnings
warnings.filterwarnings("ignore")
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import lightgbm as lgb
import optuna
from joblib import Memory
from tqdm.auto import tqdm
from statsmodels.tsa.seasonal import seasonal_decompose
import plotly.graph_objects as go

memory = Memory(".cache", verbose=0)   # кэш обучений: повторный прогон читает из .cache


def cached(func=None, *, ignore=None):
    """memory.cache со стабильным именем модуля, чтобы кэш переживал перезапуск ядра.
    По умолчанию joblib именует папку кэша по временному файлу ядра, и при рестарте кэш теряется."""
    if func is None:
        return lambda f: cached(f, ignore=ignore)
    func.__module__ = "nb_cache"
    return memory.cache(func, ignore=ignore) if ignore else memory.cache(func)
optuna.logging.set_verbosity(optuna.logging.WARNING)
sns.set_theme(style="whitegrid", palette="colorblind", rc={"figure.dpi": 110})
pd.set_option("display.float_format", lambda x: f"{x:,.4f}".replace(",", " ").rstrip("0").rstrip("."))
rng = np.random.default_rng(42)

TCOL = "week_start_date"
HORIZON = 4          # прогнозируем 4 недели вперёд
WINDOW = 150         # последние недели (для скорости расчётов на этом этапе)
DATA = Path("../hw4/data/processed/sales_weekly.parquet")

# 1. Загрузка и очистка данных

Читаем недельную панель (агрегация из дневных продаж M5 сделана на этапе подготовки данных):
units, revenue, цена, SNAP-дни, доступность, события на неделю. Чистка:

- оставляем только **полные недели** (`n_days == 7`), неполные на краях отбрасываем;
- недели **вне ассортимента** (`available_days == 0`) исключаем из обучения: это структурные
  нули, не спрос;
- берём подвыборку **FOODS** и последние `WINDOW` недель.

In [ ]:
#| label: load-clean
weekly = pd.read_parquet(DATA)
df = weekly[(weekly["cat_id"] == "FOODS") & (weekly["n_days"] == 7)].copy()
weeks = np.array(sorted(df[TCOL].unique()))
df = df[df[TCOL] > weeks[-WINDOW]].copy()
weeks = np.array(sorted(df[TCOL].unique()))

for c in ["item_id", "dept_id", "store_id", "state_id"]:
    df[c] = df[c].astype("category")

oos = (df["available_days"] == 0).mean()
zeros = (df["units"] == 0).mean()
print(f"рядов: {df['id'].nunique():,}  недель: {len(weeks)}  строк: {len(df):,}".replace(",", " "))
print(f"категория: {df['cat_id'].iat[0]}  отделов: {df['dept_id'].nunique()}  "
      f"товаров: {df['item_id'].nunique():,}".replace(",", " "))
print(f"штатов: {df['state_id'].nunique()} ({', '.join(df['state_id'].cat.categories)})  "
      f"магазинов: {df['store_id'].nunique()}")
print(f"период: {weeks[0].date()} .. {weeks[-1].date()}")
print(f"доля недель вне ассортимента (OOS): {oos:.1%}  доля нулевых недель: {zeros:.1%}")
DATA_SIG = f"{len(df)}-{weeks[-1].date()}-{len(weeks)}-w{WINDOW}"   # короткая подпись данных для ключей кэша

Пропуски на нашем уровне (товар × магазин × неделя): NaN в цене это недели, когда товара не было
в продаже. Панель полная по строкам (структурные нули, не разрывы), такие недели исключаются из
обучения.

In [ ]:
#| label: missing
na = df.isna().sum()
na = na[na > 0]
print("пропуски (NaN) по столбцам:")
print(na.to_string() if len(na) else "  нет")
full = df["id"].nunique() * df[TCOL].nunique()
print(f"\nполнота панели: {len(df):,} строк = {len(df) / full:.0%} от (ряд × неделя)".replace(",", " "))

# 2. Анализ данных (EDA)

## 2.1. Обзор данных

Сначала общий взгляд: какие колонки, типы, диапазоны числовых полей, число уникальных значений
у категориальных. `describe(include="all")`, даты без времени, пустые клетки скрыты.

In [ ]:
#| label: eda-describe
def _fmt(v):
    if isinstance(v, pd.Timestamp):
        return str(v.date())                       # дата без времени
    if pd.api.types.is_number(v):
        return "" if pd.isna(v) else f"{float(v):,.4f}".replace(",", " ").rstrip("0").rstrip(".")
    return v
df.drop(columns="wm_yr_wk").describe(include="all").T.map(_fmt)

Прогнозируем одну величину: `units` (продано штук) это целевая переменная. Остальное (выручка,
цена, SNAP-дни, доступность, события) это признаки недели, по которым строим прогноз, а не цели.
Ряд это товар (`item_id`) в магазине (`store_id`), наблюдение это неделя (`week_start_date`).

Распределения числовых полей:

In [ ]:
#| label: eda-distributions
#| code-fold: true
num = ["units", "revenue", "sell_price", "snap_days", "available_days", "event_days"]
df[num].hist(bins=40, figsize=(11, 6), color="C0")
plt.tight_layout(); plt.show()

## 2.2. Распределение спроса

Спрос на товар прерывистый: заметная доля нулевых недель и длинный правый хвост (числа ниже).
Это определяет выбор в сторону tree models с Tweedie-loss.

In [ ]:
#| label: eda-target
#| code-fold: true
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
ax[0].hist(np.log1p(df["units"]), bins=50, color="C0")
ax[0].set(title="log1p(units): пик в нуле + хвост", xlabel="log1p(units)")
share_zero = df.groupby("id", observed=True)["units"].apply(lambda s: (s == 0).mean())
ax[1].hist(share_zero, bins=30, color="C1")
ax[1].set(title="доля нулевых недель на ряд", xlabel="доля нулей")
plt.tight_layout(); plt.show()
print(f"медиана units: {df['units'].median():.0f}  среднее: {df['units'].mean():.1f}  "
      f"95-й перцентиль: {df['units'].quantile(0.95):.0f}")
print(f"доля нулевых недель: {(df['units'] == 0).mean():.1%}  "
      f"доля рядов с >50% нулевых недель: {(share_zero > 0.5).mean():.1%}")

Медиана ниже среднего, а 95-й перцентиль почти в 9 раз выше медианы: спрос низкий и сильно
скошен вправо, с тяжёлым хвостом редких крупных недель. Вместе с долей нулей это прерывистый
не-гауссов спрос, отсюда выбор Tweedie-loss.

## 2.3. Сезонность

Суммарный спрос по неделям года: виден годовой профиль (пики и спады). Сезонность кладём
в признаки через гармоники недели года.

In [ ]:
#| label: eda-season
#| code-fold: true
woy = df[TCOL].dt.isocalendar().week.astype(int)
season = df.assign(woy=woy).groupby("woy")["units"].sum()
fig, ax = plt.subplots(figsize=(9, 3))
ax.plot(season.index, season.values, color="C2")
ax.set(title="Суммарный спрос FOODS по неделе года", xlabel="неделя года", ylabel="units")
plt.tight_layout(); plt.show()

## 2.4. Иерархия

M5 имеет иерархию товар → отдел → категория и магазин → штат. Совокупная метрика WRMSSE
усредняет ошибку по этим уровням, поэтому модель должна быть точной и на агрегатах.

In [ ]:
#| label: eda-hier
#| code-fold: true
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
df.groupby("dept_id", observed=True)["units"].sum().plot.bar(ax=ax[0], color="C0")
ax[0].set(title="Спрос по отделам", ylabel="units")
df.groupby("store_id", observed=True)["units"].sum().plot.bar(ax=ax[1], color="C3")
ax[1].set(title="Спрос по магазинам")
plt.tight_layout(); plt.show()

## 2.5. Цена и промо

Прокси-промо определяем как падение цены ниже 52-недельной медианы ряда. В открытых данных
явных промо-флагов нет, поэтому это слабый сигнал, но мы его добавим как признак.

In [ ]:
#| label: eda-price
#| code-fold: true
g = df.sort_values([ "id", TCOL]).groupby("id", observed=True)["sell_price"]
med = g.transform(lambda s: s.shift(1).rolling(52, min_periods=8).median())
df["price_rel"] = (df["sell_price"] / med).astype("float32")
df["is_promo"] = (df["price_rel"] <= 0.9).astype("int8")
print(f"доля недель с прокси-промо (цена <= 0.9 от медианы): {df['is_promo'].mean():.1%}")
lift = df.groupby("is_promo", observed=True)["units"].mean()
print(f"средний units без промо: {lift.get(0, float('nan')):.2f}  с промо: {lift.get(1, float('nan')):.2f}")

# 3. Модели

## 3.1. Признаки

Дизайн без утечки: для горизонта 4 все лаги и скользящие сдвинуты на **>= 4 недели**, поэтому
одна модель прогнозирует любую из 4 будущих недель из момента прогноза, а признаки целевой
недели (цена, календарь, SNAP) известны заранее.

In [ ]:
#| label: features
def build_features(d):
    d = d.sort_values(["id", TCOL]).reset_index(drop=True)
    u = d.groupby("id", observed=True)["units"]
    for k in [4, 5, 6, 8, 12, 13, 26, 52]:        # лаги >= горизонта
        d[f"lag_{k}"] = u.shift(k).astype("float32")
    sh = u.shift(HORIZON)                          # последняя известная неделя на горизонте
    shg = sh.groupby(d["id"], observed=True)
    for k in [4, 8, 13]:
        d[f"rmean_{k}"] = shg.rolling(k, min_periods=1).mean().reset_index(level=0, drop=True).astype("float32")
    d["rstd_8"] = shg.rolling(8, min_periods=2).std().reset_index(level=0, drop=True).astype("float32")
    woy = d[TCOL].dt.isocalendar().week.astype(int).clip(1, 53)
    d["woy_sin"] = np.sin(2 * np.pi * woy / 52).astype("float32")
    d["woy_cos"] = np.cos(2 * np.pi * woy / 52).astype("float32")
    d["month"] = d[TCOL].dt.month.astype("int8")
    return d

FEATURES = ([f"lag_{k}" for k in [4, 5, 6, 8, 12, 13, 26, 52]]
            + ["rmean_4", "rmean_8", "rmean_13", "rstd_8",
               "woy_sin", "woy_cos", "month", "sell_price", "price_rel", "is_promo",
               "snap_days", "event_days"])
CAT = ["item_id", "dept_id", "store_id", "state_id"]
frame = build_features(df)
print(f"признаков: {len(FEATURES + CAT)}  строк в кадре: {len(frame):,}".replace(",", " "))

Признаки и что они означают:

| признак | что означает |
|---|---|
| `lag_4`, `lag_5`, `lag_6`, `lag_8`, `lag_12`, `lag_13`, `lag_26`, `lag_52` | продажи ряда столько недель назад; задают текущий уровень и недавнюю динамику, `lag_52` это спрос той же недели год назад. Все лаги не ближе 4 недель, чтобы на горизонте 4 недели не было утечки |
| `rmean_4`, `rmean_8`, `rmean_13` | скользящее среднее продаж за 4, 8 и 13 недель (со сдвигом на горизонт); сглаженный уровень спроса на разных окнах |
| `rstd_8` | скользящее стандартное отклонение за 8 недель; изменчивость ряда |
| `woy_sin`, `woy_cos` | неделя года через синус и косинус; непрерывная годовая сезонность без разрыва на стыке декабря и января |
| `month` | номер месяца от 1 до 12; помесячная сезонность |
| `sell_price` | цена продажи на неделе |
| `price_rel` | цена относительно 52-недельной медианы ряда; меньше 1 значит дешевле обычного |
| `is_promo` | признак скидки: цена не выше 0.9 от медианы |
| `snap_days` | число дней SNAP на неделе; это дни продовольственных талонов США, в них спрос на еду выше |
| `event_days` | число дней с событием или праздником на неделе (из календаря M5) |
| `item_id`, `dept_id`, `store_id`, `state_id` | категориальные ключи товара, отдела, магазина и штата; одна модель учится на всех рядах сразу и переносит закономерности между похожими товарами и точками |

## 3.2. Простые модели (baseline)

Три простые per-series эвристики, каждый ряд считается сам по себе:

- **наивная сезонная**: спрос той же недели год назад;
- **скользящее среднее MA-4**: среднее последних 4 недель;
- **MA × сезонность отдела**: уровень MA умножаем на сезонный множитель отдела (`dept`)
  по неделе года, извлечённый `seasonal_decompose` (устойчивее профиля по отдельному ряду).

In [ ]:
#| label: simple-models
def seasonal_naive(train, test):
    hist = train[["id", TCOL, "units"]].rename(columns={TCOL: "ly", "units": "p"})
    t = test[["id", TCOL]].copy()
    t["ly"] = t[TCOL] - pd.Timedelta(days=364)
    t = t.merge(hist, on=["id", "ly"], how="left")
    t["pred"] = t["p"].fillna(0.0)
    return t[["id", TCOL, "pred"]]

def ma_level(train, k):
    wk = sorted(train[TCOL].unique())[-k:]
    return (train[train[TCOL].isin(wk)].groupby("id", observed=True)["units"]
            .mean().rename("lvl").reset_index())

def moving_avg(train, test, k=4):
    t = test[["id", TCOL]].merge(ma_level(train, k), on="id", how="left")
    t["pred"] = t["lvl"].fillna(0.0)
    return t[["id", TCOL, "pred"]]

def seasonal_ma(train, test, k=8):
    """MA-уровень ряда × сезонный множитель отдела (seasonal_decompose по неделе года)."""
    lvl = ma_level(train, k)
    rows = []
    for dept, g in train.groupby("dept_id", observed=True):
        s = g.groupby(TCOL, observed=True)["units"].sum().sort_index()
        seas = seasonal_decompose(s.values, model="multiplicative", period=52).seasonal
        woy = pd.Series(s.index).dt.isocalendar().week.astype(int).values
        factor = pd.Series(seas, index=woy).groupby(level=0).mean()
        rows += [{"dept_id": dept, "woy": w, "sidx": v} for w, v in factor.items()]
    sidx = pd.DataFrame(rows)
    t = test[["id", "dept_id", TCOL]].copy()
    t["woy"] = t[TCOL].dt.isocalendar().week.astype(int)
    t = t.merge(lvl, on="id", how="left").merge(sidx, on=["dept_id", "woy"], how="left")
    t["sidx"] = t["sidx"].fillna(1.0).clip(0, 3)
    t["pred"] = (t["lvl"].fillna(0.0) * t["sidx"]).clip(lower=0)
    return t[["id", TCOL, "pred"]]

## 3.3. Усложнённая модель: LightGBM

Берём градиентный бустинг и учим одну модель сразу на всех товарах и магазинах. Плоский
LightGBM (обычный L2-loss, без поправок) на этих данных не лучше простых моделей. Качество
дают две правки, вклад каждой считаем в разделе 5.1:

1. **Tweedie-loss** под прерывистый спрос. Tweedie-распределение (составное Пуассон-Гамма)
   описывает смесь точных нулей и положительной скошенной части, то есть ровно наш спрос; в M5
   это одна из причин сильной работы tree models. Разбор:
   [Forecasting intermittent time series with Gaussian Processes and Tweedie likelihood](https://arxiv.org/abs/2502.19086).
2. **Калибровка смещения**: Tweedie систематически перепрогнозирует; умножаем прогноз на
   `sum(факт) / sum(прогноз)` со свежего блока до момента прогноза. Без неё бустинг проигрывает
   на агрегатах, где WRMSSE усредняет уровни иерархии поровну.

Дополнительно: каждое наблюдение входит в loss с весом `log1p(выручки)`, поэтому ряды с большой
выручкой модель оптимизирует сильнее (как и WRMSSE). Гиперпараметры подбираем Optuna.

In [ ]:
#| label: lgb-funcs
DEFAULT = dict(objective="tweedie", tweedie_variance_power=1.1, learning_rate=0.05,
               num_leaves=63, min_data_in_leaf=100, feature_fraction=0.8,
               bagging_fraction=0.8, bagging_freq=1, lambda_l2=0.1,
               num_threads=0, verbose=-1, force_col_wise=True, seed=42)

@cached
def lgb_train(tr, params, rounds=400):
    p = {**DEFAULT, **params}
    if p.get("objective") != "tweedie":           # L2 и прочие не используют tweedie_variance_power
        p.pop("tweedie_variance_power", None)
    w = np.log1p(tr["revenue"].clip(lower=0).fillna(0))
    dset = lgb.Dataset(tr[FEATURES + CAT], label=tr["units"], weight=w,
                       categorical_feature=CAT, free_raw_data=False)
    return lgb.train(p, dset, num_boost_round=rounds, callbacks=[lgb.log_evaluation(0)])

def lgb_predict(model, rows, calib_rows=None):
    pred = np.clip(model.predict(rows[FEATURES + CAT]), 0, None)
    out = rows[["id", TCOL]].copy()
    out["pred"] = pred
    if calib_rows is not None:                    # калибровка смещения
        cp = np.clip(model.predict(calib_rows[FEATURES + CAT]), 0, None).sum()
        factor = calib_rows["units"].sum() / max(cp, 1e-9)
        out["pred"] *= factor
    # маска вне ассортимента
    out = out.merge(rows[["id", TCOL, "available_days"]], on=["id", TCOL])
    out["pred"] = np.where(out["available_days"] == 0, 0.0, out["pred"])
    return out[["id", TCOL, "pred"]]

## 3.4. AutoML для сравнения (StatsForecast)

Добавляем готовую модель автоматического прогноза из библиотеки StatsForecast (Nixtla): **IMAPA**,
метод для прерывистого спроса, библиотека сама оценивает его параметры по ряду. Его прогноз
участвует в сравнении моделей ниже (раздел 4.3) строкой «StatsForecast IMAPA».

In [ ]:
#| label: statsforecast
@cached(ignore=["train_raw"])
def sf_forecast(train_raw, test_weeks, sig):
    from statsforecast import StatsForecast
    from statsforecast.models import IMAPA
    sdf = train_raw[["id", TCOL, "units"]].rename(columns={"id": "unique_id", TCOL: "ds", "units": "y"})
    fc = StatsForecast(models=[IMAPA()], freq="W-SAT", n_jobs=1).forecast(df=sdf, h=HORIZON)
    col = [c for c in fc.columns if c not in ("unique_id", "ds")][0]
    fc = fc.copy()
    fc["h"] = fc.groupby("unique_id").cumcount()
    fc[TCOL] = fc["h"].map({i: test_weeks[i] for i in range(HORIZON)})
    fc["pred"] = fc[col].clip(lower=0)
    return fc.rename(columns={"unique_id": "id"})[["id", TCOL, "pred"]]

# 4. Метрики и валидация

## 4.1. Метрики

- **RMSSE** на ряд: корень из MSE прогноза, нормированный на средний квадрат недельного
  шага `y_t - y_{t-1}` по train (ошибка прогноза «как неделю назад»).
- **WRMSSE** (совокупная): взвешенный RMSSE по уровням иерархии M5 (для выборки FOODS это
  9 уровней: тотал, штат, магазин, отдел, их пересечения, товар, item × store), веса по доле
  выручки; уровни усредняются поровну.
- **MASE**: MAE прогноза, делённый на средний `|y_t - y_{t-1}|` по train.
- **Bias**: относительное систематическое смещение `(sum pred - sum факт) / sum факт`.

Все метрики считаем на тесте (валидационные фолды, данные после момента прогноза). Train-метрики
смотрят отдельно для контроля пере- и недообучения, но база это тест.

In [ ]:
#| label: metrics
LEVELS = [("L1", []), ("L2", ["state_id"]), ("L3", ["store_id"]), ("L4", ["dept_id"]),
          ("L5", ["state_id", "dept_id"]), ("L6", ["store_id", "dept_id"]),
          ("L7", ["item_id"]), ("L8", ["item_id", "state_id"]), ("L9", ["id"])]
LVL_NAMES = {"L1": "тотал", "L2": "штат", "L3": "магазин", "L4": "отдел", "L5": "штат × отдел",
             "L6": "магазин × отдел", "L7": "товар", "L8": "товар × штат", "L9": "товар × магазин"}

def _agg(d, keys, val):
    if not keys:
        g = d.groupby(TCOL, observed=True)[val].sum().reset_index(); g["k"] = "T"; return g
    g = d.groupby(keys + [TCOL], observed=True)[val].sum().reset_index()
    g["k"] = g[keys].astype(str).agg("|".join, axis=1)
    return g[["k", TCOL, val]]

def _scale(tr_lvl):
    s = tr_lvl.sort_values(["k", TCOL]).copy()
    s["d2"] = s.groupby("k", observed=True)["v"].diff() ** 2
    sc = s.groupby("k", observed=True)["d2"].mean()
    return sc[sc > 0]

def make_scorer(train):
    """Кэширует scale и веса одного train-фолда, чтобы дёшево оценивать много моделей."""
    last = sorted(train[TCOL].unique())[-4:]
    levels = []
    for _, keys in LEVELS:
        sc = _scale(_agg(train, keys, "units").rename(columns={"units": "v"}))
        if sc.empty:
            continue
        wt = _agg(train[train[TCOL].isin(last)], keys, "revenue").groupby("k")["revenue"].sum()
        levels.append((keys, sc, wt))
    ts = train.sort_values(["id", TCOL]); dd = ts.groupby("id", observed=True)["units"].diff()
    s2 = (dd ** 2).groupby(ts["id"]).mean(); s2 = s2[s2 > 0]
    s1 = dd.abs().groupby(ts["id"]).mean(); s1 = s1[s1 > 0]

    def score(test_actual, pred, name):
        t = test_actual.merge(pred, on=["id", TCOL], how="left"); t["pred"] = t["pred"].fillna(0.0)
        ws = []
        for keys, sc, wt in levels:                          # WRMSSE по уровням иерархии
            a = _agg(t, keys, "units").rename(columns={"units": "va"})
            p = _agg(t.assign(units=t["pred"]), keys, "units").rename(columns={"units": "vp"})
            m = a.merge(p, on=["k", TCOL], how="left").fillna(0.0)
            mse = ((m["va"] - m["vp"]) ** 2).groupby(m["k"]).mean()
            idx = sc.index.intersection(mse.index); rmsse = np.sqrt(mse[idx] / sc[idx])
            wk = wt.reindex(rmsse.index).fillna(0.0)
            if wk.sum() > 0:
                ws.append((rmsse * wk).sum() / wk.sum())
        se = ((t["units"] - t["pred"]) ** 2).groupby(t["id"]).mean()     # RMSSE / MASE на ряд
        ae = (t["units"] - t["pred"]).abs().groupby(t["id"]).mean()
        i2 = s2.index.intersection(se.index); i1 = s1.index.intersection(ae.index)
        a_sum = t["units"].sum()
        return {"модель": name, "WRMSSE": round(float(np.mean(ws)), 4),
                "RMSSE": round(float(np.sqrt(se[i2] / s2[i2]).mean()), 4),
                "MASE": round(float((ae[i1] / s1[i1]).mean()), 4),
                "Bias": round(float((t["pred"].sum() - a_sum) / a_sum) if a_sum > 0 else np.nan, 4)}
    return score

## 4.2. Подбор гиперпараметров (Optuna)

Подбираем гиперпараметры LightGBM по валидационной WRMSSE на блоке до тестовых фолдов (без утечки
в тест). Подбор кэшируется (`joblib.Memory`), поэтому повторный прогон берёт параметры из кэша и не
перебирает варианты заново.

In [ ]:
#| label: optuna
val_origin = weeks[-(HORIZON * 4)]                # валидация перед тестовыми фолдами
val_w = list(weeks[(np.where(weeks == val_origin)[0][0] + 1):][:HORIZON])
tr_val = frame[(frame[TCOL] <= val_origin) & (frame["available_days"] > 0)]
te_val = frame[frame[TCOL].isin(val_w)]
cal_val = frame[frame[TCOL].isin(list(weeks[weeks <= val_origin][-HORIZON:]))]
val_actual = df[df[TCOL].isin(val_w)]

# ключ кэша по подписи данных (sig); большие кадры исключаем из ключа через ignore
@cached(ignore=["tr_val", "te_val", "cal_val", "val_actual", "scorer_train"])
def tune_lgb(tr_val, te_val, cal_val, val_actual, scorer_train, sig, n_trials=10):
    val_scorer = make_scorer(scorer_train)

    def objective(trial):
        p = dict(num_leaves=trial.suggest_int("num_leaves", 31, 255),
                 min_data_in_leaf=trial.suggest_int("min_data_in_leaf", 50, 500),
                 feature_fraction=trial.suggest_float("feature_fraction", 0.6, 1.0),
                 learning_rate=trial.suggest_float("learning_rate", 0.03, 0.08, log=True))
        pred = lgb_predict(lgb_train(tr_val, p, rounds=300), te_val, calib_rows=cal_val)
        return val_scorer(val_actual, pred, "opt")["WRMSSE"]

    study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=42))
    study.optimize(objective, n_trials=n_trials, show_progress_bar=True)
    return study.best_params, study.best_value

best_params, best_value = tune_lgb(tr_val, te_val, cal_val, val_actual, df[df[TCOL] <= val_origin], DATA_SIG)
print(f"лучшие параметры: {best_params}")
print(f"валидационная WRMSSE: {best_value:.4f}")

## 4.3. Walk-forward и сравнение моделей

Валидация это time-series split (walk-forward): без перемешивания, на каждом фолде учимся только
на прошлом (`train <= origin`) и проверяем на следующих 4 неделях. 3 фолда, метрики усредняем по
ним. Прогон по фолдам кэшируется: повторный запуск берёт готовые метрики из кэша.

In [ ]:
#| label: walk-forward
N_FOLDS = 3
folds = []
for k in range(N_FOLDS):
    e = len(weeks) - 1 - k
    o = e - HORIZON
    folds.append({"origin": weeks[o], "test_w": list(weeks[o + 1:e + 1])})
folds = list(reversed(folds))

# ключ кэша по best_params и подписи данных (sig); большие кадры исключаем из ключа через ignore
@cached(ignore=["df", "frame", "weeks", "folds"])
def run_walk_forward(df, frame, weeks, folds, best_params, sig):
    rows = []
    for fi, fold in enumerate(tqdm(folds, desc="фолды"), 1):
        train_raw = df[df[TCOL] <= fold["origin"]]
        test_raw = df[df[TCOL].isin(fold["test_w"])]
        tr = frame[(frame[TCOL] <= fold["origin"]) & (frame["available_days"] > 0)]
        cal = frame[frame[TCOL].isin(list(weeks[weeks <= fold["origin"]][-HORIZON:]))]
        te = frame[frame[TCOL].isin(fold["test_w"])]
        scorer = make_scorer(train_raw)
        preds = {
            "наивная сезонная": seasonal_naive(train_raw, test_raw),
            "MA-4": moving_avg(train_raw, test_raw),
            "MA × сезонность": seasonal_ma(train_raw, test_raw),
            "LightGBM Tweedie": lgb_predict(lgb_train(tr, {}), te, calib_rows=cal),
            "LightGBM Tweedie + Optuna": lgb_predict(lgb_train(tr, best_params), te, calib_rows=cal),
            "StatsForecast IMAPA": sf_forecast(train_raw, fold["test_w"], sig),
        }
        for name, p in preds.items():
            r = scorer(test_raw, p, name); r["fold"] = fi
            rows.append(r)
    return pd.DataFrame(rows)

perfold = run_walk_forward(df, frame, weeks, folds, best_params, DATA_SIG)

In [ ]:
#| label: results
#| code-fold: true
summary = (perfold.groupby("модель")[["WRMSSE", "RMSSE", "MASE", "Bias"]]
           .mean().round(4).sort_values("WRMSSE"))
summary

WRMSSE по каждому валидационному фолду (видно устойчивость порядка моделей):

In [ ]:
#| label: per-fold
#| code-fold: true
perfold.pivot(index="fold", columns="модель", values="WRMSSE").round(4)

In [ ]:
#| label: results-plot
#| code-fold: true
#| fig-cap: "WRMSSE по моделям (среднее по 3 фолдам). Ниже лучше; 1.0 = уровень наивного шага."
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.barh(summary.index, summary["WRMSSE"], color=sns.color_palette("crest", len(summary)))
ax.axvline(1.0, color="C3", ls="--", lw=1, label="RMSSE = 1")
ax.set(xlabel="WRMSSE (ниже лучше)", title="Сравнение моделей, walk-forward")
ax.legend(); plt.tight_layout(); plt.show()

# 5. Анализ результатов

In [ ]:
#| label: analysis
#| code-fold: true
best_model = summary.index[0]
simple = [m for m in ["наивная сезонная", "MA-4", "MA × сезонность"] if m in summary.index]
base = summary.loc[simple, "WRMSSE"].idxmin()                     # сильнейший простой baseline
gain = (summary.loc[base, "WRMSSE"] - summary.loc[best_model, "WRMSSE"]) / summary.loc[base, "WRMSSE"] * 100
print(f"лучшая модель: {best_model} (WRMSSE {summary.loc[best_model, 'WRMSSE']})")
print(f"лучший простой baseline: {base} (WRMSSE {summary.loc[base, 'WRMSSE']})")
print(f"улучшение над лучшим простым baseline: {gain:+.1f}%")
print(f"Bias лучшей модели после калибровки: {summary.loc[best_model, 'Bias']:+.4f}")

## 5.1. Из чего складывается качество LightGBM

Плоский LightGBM (L2-loss, без калибровки) не лучше простых моделей. Качество дают две правки:
Tweedie-loss под прерывистый спрос и калибровка смещения. Вклад каждой на последнем фолде:

In [ ]:
#| label: ablation
fold = folds[-1]
tr_raw = df[df[TCOL] <= fold["origin"]]; te_raw = df[df[TCOL].isin(fold["test_w"])]
trf = frame[(frame[TCOL] <= fold["origin"]) & (frame["available_days"] > 0)]
calf = frame[frame[TCOL].isin(list(weeks[weeks <= fold["origin"]][-HORIZON:]))]
tef = frame[frame[TCOL].isin(fold["test_w"])]
sc = make_scorer(tr_raw)
m_l2 = lgb_train(trf, {"objective": "regression"})
m_tw = lgb_train(trf, {})
m_opt = lgb_train(trf, best_params)
abl = pd.DataFrame([
    sc(te_raw, lgb_predict(m_l2, tef), "1. L2-loss, без калибровки"),
    sc(te_raw, lgb_predict(m_tw, tef), "2. + Tweedie-loss"),
    sc(te_raw, lgb_predict(m_tw, tef, calib_rows=calf), "3. + калибровка смещения"),
    sc(te_raw, lgb_predict(m_opt, tef, calib_rows=calf), "4. + подбор Optuna"),
])[["модель", "WRMSSE", "Bias"]]
abl["dWRMSSE"] = abl["WRMSSE"].diff().round(4)
abl

Калибровка даёт основной скачок WRMSSE: убирает систематический перепрогноз и приводит Bias к
нулю. Совокупная метрика усредняет уровни иерархии поровну, поэтому смещение на агрегатах решает.
Tweedie-loss добавляет поверх L2, подбор Optuna даёт небольшой прирост.

## 5.2. Как выглядит прогноз

Один ряд с большим объёмом: история, факт на горизонте и прогнозы MA-4 и LightGBM. Точки после
вертикальной линии (момент прогноза) это предсказание на 4 недели вперёд.

In [ ]:
#| label: forecast-example
#| code-fold: true
ex = tr_raw.groupby("id", observed=True)["units"].sum().sort_values().index[-1]
hist_ex = df[(df["id"] == ex) & (df[TCOL] > fold["origin"] - pd.Timedelta(weeks=26))]
p_ma = moving_avg(tr_raw, te_raw)
p_lgb = lgb_predict(m_opt, tef, calib_rows=calf)
fig = go.Figure()
fig.add_scatter(x=hist_ex[TCOL], y=hist_ex["units"], name="факт", mode="lines+markers",
                line={"color": "#37474f", "width": 2.2})
for nm, p, col in [("MA-4", p_ma, "#2980b9"), ("LightGBM Tweedie + Optuna", p_lgb, "#e67e22")]:
    pe = p[p["id"] == ex]
    fig.add_scatter(x=pe[TCOL], y=pe["pred"], name=nm, mode="lines+markers",
                    line={"color": col, "width": 2, "dash": "dash"})
fig.add_shape(type="line", x0=fold["origin"], x1=fold["origin"], yref="paper", y0=0, y1=1,
              line={"color": "grey", "width": 1, "dash": "dot"})
fig.update_layout(title=f"Прогноз и факт: {ex}", template="plotly_white", height=380,
                  yaxis_title="units", legend={"orientation": "h", "y": -0.25},
                  margin={"l": 55, "r": 20, "t": 46, "b": 70})
fig.show()

## 5.3. Выводы

- Наивная сезонная слабая: ассортимент меняется, год назад части рядов не было.
- MA-4 сильная простая планка: свежее среднее почти несмещено, на агрегатах WRMSSE точно.
- MA × сезонность отдела на уровне MA-4: сезонный множитель отдела устойчив; профиль по
  отдельному ряду был заметно хуже из-за шума.
- LightGBM лучший. Из таблицы выше виден вклад правок: основной прирост от калибровки смещения,
  Tweedie-loss и подбор Optuna добавляют поверх.

# 6. Итог по метрикам качества

Оценивали четырьмя метриками, главная из них совокупная:

- **WRMSSE** (совокупная): взвешенный RMSSE по уровням иерархии M5.
- **RMSSE**: масштабированная среднеквадратичная ошибка на ряд.
- **MASE**: масштабированная средняя абсолютная ошибка на ряд.
- **Bias**: систематическое смещение прогноза.

Итоговые числа лучшей модели (среднее по валидационным фолдам) и полная сводка:

In [ ]:
#| label: final-metrics
#| code-fold: true
best = summary.index[0]
simple = [m for m in ["наивная сезонная", "MA-4", "MA × сезонность"] if m in summary.index]
base = summary.loc[simple, "WRMSSE"].idxmin()
imp = (summary.loc[base, "WRMSSE"] - summary.loc[best, "WRMSSE"]) / summary.loc[base, "WRMSSE"] * 100
print("Итоговые метрики качества (среднее по валидационным фолдам):")
print(f"  лучшая модель:        {best}")
print(f"  WRMSSE (совокупная):  {summary.loc[best, 'WRMSSE']}")
print(f"  RMSSE:                {summary.loc[best, 'RMSSE']}")
print(f"  MASE:                 {summary.loc[best, 'MASE']}")
print(f"  Bias:                 {summary.loc[best, 'Bias']:+.4f}")
print(f"  улучшение над лучшим простым baseline ({base}): {imp:+.1f}%")
summary

Ориентир M5: в соревновании Kaggle M5 Accuracy лучшие решения по WRMSSE были около 0.52-0.54. Их
метрика считается иначе (дневной горизонт 28 дней, все 42 840 рядов трёх категорий, 12 уровней
иерархии), поэтому с нашим числом напрямую не сравнивается: у нас недельный горизонт 4 недели,
выборка FOODS, 9 уровней, последние 150 недель. Это внешний ориентир порядка величины.

## 6.1. Тепловая карта: WRMSSE по уровням иерархии и моделям

WRMSSE каждой модели на каждом уровне агрегации иерархии M5 (последний фолд). Уровни от верхнего к
нижнему: тотал (все продажи), штат, магазин, отдел, штат × отдел, магазин × отдел, товар (по всем
магазинам), товар × штат, товар × магазин (самый нижний). Зелёное лучше, красное хуже; 1.0 это
уровень наивного недельного шага.

In [ ]:
#| label: heatmap
#| code-fold: true
last_fold = folds[-1]
tr_raw_h = df[df[TCOL] <= last_fold["origin"]]
te_raw_h = df[df[TCOL].isin(last_fold["test_w"])]
tr_h = frame[(frame[TCOL] <= last_fold["origin"]) & (frame["available_days"] > 0)]
cal_h = frame[frame[TCOL].isin(list(weeks[weeks <= last_fold["origin"]][-HORIZON:]))]
te_h = frame[frame[TCOL].isin(last_fold["test_w"])]
builders = {
    "наивная сезонная": lambda: seasonal_naive(tr_raw_h, te_raw_h),
    "MA-4": lambda: moving_avg(tr_raw_h, te_raw_h),
    "MA × сезонность": lambda: seasonal_ma(tr_raw_h, te_raw_h),
    "LightGBM Tweedie": lambda: lgb_predict(lgb_train(tr_h, {}), te_h, calib_rows=cal_h),
    "LightGBM Tweedie + Optuna": lambda: lgb_predict(lgb_train(tr_h, best_params), te_h, calib_rows=cal_h),
    "StatsForecast IMAPA": lambda: sf_forecast(tr_raw_h, last_fold["test_w"], DATA_SIG),
}
model_preds = {nm: build() for nm, build in tqdm(builders.items(), desc="модели")}

def level_wrmsse(train, test, pred):
    t = test.merge(pred, on=["id", TCOL], how="left"); t["pred"] = t["pred"].fillna(0.0)
    last4 = sorted(train[TCOL].unique())[-4:]; out = {}
    for name, keys in LEVELS:
        base = _agg(train, keys, "units").rename(columns={"units": "v"}).sort_values(["k", TCOL])
        sc = (base.groupby("k", observed=True)["v"].diff() ** 2).groupby(base["k"]).mean(); sc = sc[sc > 0]
        if sc.empty:
            continue
        a = _agg(t, keys, "units").rename(columns={"units": "va"})
        p = _agg(t.assign(units=t["pred"]), keys, "units").rename(columns={"units": "vp"})
        m = a.merge(p, on=["k", TCOL], how="left").fillna(0.0)
        mse = ((m["va"] - m["vp"]) ** 2).groupby(m["k"]).mean(); idx = sc.index.intersection(mse.index)
        rmsse = np.sqrt(mse[idx] / sc[idx])
        wt = _agg(train[train[TCOL].isin(last4)], keys, "revenue").groupby("k")["revenue"].sum().reindex(rmsse.index).fillna(0.0)
        out[name] = round((rmsse * wt).sum() / max(wt.sum(), 1e-9), 3)
    return out

heat = pd.DataFrame({nm: level_wrmsse(tr_raw_h, te_raw_h, pr) for nm, pr in model_preds.items()})
fig = go.Figure(go.Heatmap(
    z=heat.values, x=list(heat.columns), y=[LVL_NAMES[l] for l in heat.index],
    colorscale="RdYlGn_r", zmid=1.0, colorbar={"title": "WRMSSE"},
    text=heat.round(2).values, texttemplate="%{text}", textfont={"size": 11},
    hovertemplate="%{y}<br>%{x}<br>WRMSSE %{z}<extra></extra>"))
fig.update_layout(title="WRMSSE по уровням иерархии и моделям (зелёное лучше, последний фолд)",
                  template="plotly_white", height=430, yaxis={"autorange": "reversed"},
                  margin={"l": 120, "r": 20, "t": 56, "b": 80})
fig.show()

По карте: на агрегатах (тотал, штат, магазин, отдел) бустинг заметно точнее, а на уровне отдельного
товара простое скользящее среднее не уступает - там преобладает шум ряда. Совокупную WRMSSE
итоговая модель выигрывает за счёт верхних уровней иерархии.

# 7. Улучшение модели

Показываем переход от простого baseline к улучшенной модели и вклад каждого шага по осям:
архитектура (LightGBM), постобработка (калибровка, квантили), признаки (события). Считаем на
последнем фолде, как в разделе 5.1.

In [ ]:
#| label: hw7-helpers
#| code-fold: true
@cached
def fit_lgb(tr, te, cal, feats, params=None):
    """LightGBM на наборе признаков feats с калибровкой смещения."""
    w = np.log1p(tr["revenue"].clip(lower=0).fillna(0))
    cats = [c for c in CAT if c in feats]
    ds = lgb.Dataset(tr[feats], label=tr["units"], weight=w, categorical_feature=cats, free_raw_data=False)
    m = lgb.train({**DEFAULT, **(params or {})}, ds, num_boost_round=400, callbacks=[lgb.log_evaluation(0)])
    pred = np.clip(m.predict(te[feats]), 0, None)
    pred = pred * (cal["units"].sum() / max(np.clip(m.predict(cal[feats]), 0, None).sum(), 1e-9))
    out = te[["id", TCOL, "available_days"]].copy()
    out["pred"] = np.where(out["available_days"] == 0, 0.0, pred)
    return out[["id", TCOL, "pred"]]


def add_event_features(d):
    """Недельные счётчики дней по 4 типам событий из календаря M5."""
    cal = pd.read_parquet(DATA.parent / "calendar.parquet")
    types = ["Cultural", "National", "Religious", "Sporting"]
    cols = [f"ev_{t.lower()}" for t in types]
    ev = pd.concat([cal[["wm_yr_wk", c]].rename(columns={c: "type"})
                    for c in ["event_type_1", "event_type_2"]]).dropna()
    tbl = cal[["wm_yr_wk"]].drop_duplicates().set_index("wm_yr_wk")
    for t, c in zip(types, cols):
        tbl[c] = ev[ev["type"] == t].groupby("wm_yr_wk").size().reindex(tbl.index).fillna(0).astype("int8")
    out = d.merge(tbl.reset_index(), on="wm_yr_wk", how="left")
    out[cols] = out[cols].fillna(0).astype("int8")
    return out, cols

## 7.1. От простого baseline к улучшенной модели

In [ ]:
#| label: hw7-ladder
fold = folds[-1]
train_raw = df[df[TCOL] <= fold["origin"]]
test_raw = df[df[TCOL].isin(fold["test_w"])]
tr = frame[(frame[TCOL] <= fold["origin"]) & (frame["available_days"] > 0)]
cal = frame[frame[TCOL].isin(list(weeks[weeks <= fold["origin"]][-HORIZON:]))]
te = frame[frame[TCOL].isin(fold["test_w"])]
scorer = make_scorer(train_raw)

frame_ev, EV = add_event_features(frame)
tr_ev = frame_ev[(frame_ev[TCOL] <= fold["origin"]) & (frame_ev["available_days"] > 0)]
cal_ev = frame_ev[frame_ev[TCOL].isin(list(weeks[weeks <= fold["origin"]][-HORIZON:]))]
te_ev = frame_ev[frame_ev[TCOL].isin(fold["test_w"])]

pred_ma = moving_avg(train_raw, test_raw)
pred_lgb = fit_lgb(tr, te, cal, FEATURES + CAT)
pred_final = fit_lgb(tr_ev, te_ev, cal_ev, FEATURES + EV + CAT)
def wrmsse(pred):
    return scorer(test_raw, pred, "")["WRMSSE"]

steps = pd.DataFrame([
    {"шаг": "MA-4 (простой baseline)", "ось": "baseline", "WRMSSE": wrmsse(pred_ma)},
    {"шаг": "LightGBM (Tweedie + калибровка)", "ось": "архитектура + постобработка",
     "WRMSSE": wrmsse(pred_lgb)},
    {"шаг": "+ признаки событий", "ось": "FE", "WRMSSE": wrmsse(pred_final)},
])
steps["Δ к пред."] = steps["WRMSSE"].diff().round(4)
steps

Главный прирост даёт переход от простого MA к LightGBM (архитектура и калибровка смещения,
разложение в разделе 5.1). Признаки событий добавляют небольшой выигрыш по FE-оси.

## 7.2. Постобработка: квантильный прогноз P10/P50/P90

Точечный прогноз дополняем интервалом: три LightGBM с pinball-loss (квантильная функция потерь).
Pinball-loss штрафует ошибку асимметрично: при alpha=0.9 перепрогноз дешевле недопрогноза, поэтому
минимум достигается на P90 и модель учит нужный квантиль напрямую. Верхний квантиль P90 задаёт
уровень под страховой запас. Качество интервала измеряем покрытием: это доля недель, где факт
оказался не выше прогноза квантиля. Для P90 покрытие должно быть около 0.9, для P50 около 0.5; чем
ближе покрытие к уровню квантиля, тем точнее откалиброван интервал.

In [ ]:
#| label: hw7-quantiles
#| code-fold: true
@cached
def fit_quantile(tr, te, feats, alpha):
    p = {k: v for k, v in DEFAULT.items() if k != "tweedie_variance_power"}
    p.update(objective="quantile", alpha=alpha)
    w = np.log1p(tr["revenue"].clip(lower=0).fillna(0))
    cats = [c for c in CAT if c in feats]
    ds = lgb.Dataset(tr[feats], label=tr["units"], weight=w, categorical_feature=cats, free_raw_data=False)
    m = lgb.train(p, ds, num_boost_round=400, callbacks=[lgb.log_evaluation(0)])
    out = te[["id", TCOL]].copy()
    out["pred"] = np.clip(m.predict(te[feats]), 0, None)
    return out

qt = test_raw[["id", TCOL, "units"]].copy()
for q in [0.1, 0.5, 0.9]:
    qt = qt.merge(fit_quantile(tr, te, FEATURES + CAT, q).rename(columns={"pred": f"q{q}"}),
                  on=["id", TCOL])
pd.DataFrame([{"квантиль": f"P{int(q * 100)}", "цель": q,
               "покрытие": round((qt["units"] <= qt[f"q{q}"]).mean(), 3)} for q in [0.1, 0.5, 0.9]])

## 7.3. Разрез метрик финальной модели по уровням иерархии

WRMSSE финальной модели по каждому уровню агрегации: видно, где модель точна (агрегаты) и где
сложнее (отдельный товар).

In [ ]:
#| label: hw7-levels
#| code-fold: true
t = test_raw.merge(pred_final, on=["id", TCOL])
last = sorted(train_raw[TCOL].unique())[-4:]
rows = []
for name, keys in LEVELS:
    base = _agg(train_raw, keys, "units").rename(columns={"units": "v"}).sort_values(["k", TCOL])
    sc = (base.groupby("k", observed=True)["v"].diff() ** 2).groupby(base["k"]).mean()
    sc = sc[sc > 0]
    if sc.empty:
        continue
    a = _agg(t, keys, "units").rename(columns={"units": "va"})
    p = _agg(t.assign(units=t["pred"]), keys, "units").rename(columns={"units": "vp"})
    m = a.merge(p, on=["k", TCOL], how="left").fillna(0.0)
    mse = ((m["va"] - m["vp"]) ** 2).groupby(m["k"]).mean()
    idx = sc.index.intersection(mse.index)
    rmsse = np.sqrt(mse[idx] / sc[idx])
    wt = _agg(train_raw[train_raw[TCOL].isin(last)], keys, "revenue").groupby("k")["revenue"].sum()
    wt = wt.reindex(rmsse.index).fillna(0.0)
    rows.append({"уровень": LVL_NAMES[name], "WRMSSE": round((rmsse * wt).sum() / max(wt.sum(), 1e-9), 4)})
pd.DataFrame(rows)

Точность растёт с агрегацией: на тотале и по магазинам WRMSSE ниже, на уровне отдельного товара
выше. Шум прерывистого ряда усредняется на агрегатах.

# 8. Реальный прогноз вперёд и интерактивный просмотр

Финальную модель (LightGBM Tweedie + Optuna с калибровкой) обучаем на всех неделях и прогнозируем
следующие 4 недели, для которых факта ещё нет. Признаки будущих недель собираем без утечки: лаги
считаются из реальной истории (все лаги сдвинуты на 4 недели и более), SNAP и события известны
заранее из календаря, цена перенесена с последней известной недели.

## 8.1. Прогноз на 4 недели вперёд

In [ ]:
#| label: real-forecast
tr_all = frame[frame["available_days"] > 0]                       # обучение на всех неделях
m_final = lgb_train(tr_all, best_params)
cal_final = frame[frame[TCOL].isin(weeks[-HORIZON:])]             # блок калибровки смещения

cal = pd.read_parquet(DATA.parent / "calendar.parquet")          # будущие недели из календаря
calw = (cal.groupby("wm_yr_wk")
        .agg(**{TCOL: ("date", "min"), "event_days": ("has_event", "sum"),
                "snap_CA": ("snap_CA", "sum"), "snap_TX": ("snap_TX", "sum"),
                "snap_WI": ("snap_WI", "sum")})
        .reset_index())
future = calw[calw[TCOL] > weeks[-1]].sort_values(TCOL).head(HORIZON)

series = df[["id", "item_id", "dept_id", "store_id", "state_id"]].drop_duplicates()
last = df.sort_values(TCOL).groupby("id", observed=True).tail(1).set_index("id")
fut = series.merge(future, how="cross")
fut["snap_days"] = np.select(
    [fut["state_id"] == "CA", fut["state_id"] == "TX", fut["state_id"] == "WI"],
    [fut["snap_CA"], fut["snap_TX"], fut["snap_WI"]], default=0).astype("int8")
fut["event_days"] = fut["event_days"].astype("int8")
fut["available_days"] = 7
fut["units"] = np.nan
fut["sell_price"] = fut["id"].map(last["sell_price"]).astype("float32")
fut["price_rel"] = fut["id"].map(last["price_rel"]).astype("float32")
fut["is_promo"] = fut["id"].map(last["is_promo"]).astype("int8")

cols = ["id", "item_id", "dept_id", "store_id", "state_id", "wm_yr_wk", TCOL,
        "units", "sell_price", "snap_days", "available_days", "event_days", "price_rel", "is_promo"]
allrows = pd.concat([df[cols], fut[cols]], ignore_index=True)
for c in CAT:
    allrows[c] = pd.Categorical(allrows[c], categories=df[c].cat.categories)
allframe = build_features(allrows)
fut_feat = allframe[allframe[TCOL].isin(future[TCOL])]
real_fc = lgb_predict(m_final, fut_feat, calib_rows=cal_final)

print("прогноз на недели:", ", ".join(str(w.date()) for w in future[TCOL]))
print(f"суммарный спрос FOODS (прогноз вперёд): {real_fc['pred'].sum():,.0f}".replace(",", " "))
real_fc.head()

## 8.2. Итоговая модель: из чего она и почему точнее

Итоговая модель это одна глобальная LightGBM, обученная сразу на всех рядах item × store, с прямым
прогнозом на 4 недели. Признаки: лаги спроса (включая годовой), скользящие среднее и разброс,
сезонность неделей года и месяцем, цена и признак скидки, SNAP и события, категориальные ключи
иерархии (товар, отдел, магазин, штат).

Почему она точнее простых моделей и AutoML:

1. Общее обучение по всем рядам: переносит закономерности между похожими товарами и магазинами,
   прерывистый ряд получает прогноз за счёт похожих. Per-series методы (наивная, MA) и AutoML IMAPA
   так не умеют.
2. Tweedie-loss под прерывистый спрос (нули и скошенная положительная часть).
3. Калибровка смещения убирает систематический перепрогноз - главный вклад в WRMSSE, который
   усредняет уровни иерархии и чувствителен к смещению на агрегатах.
4. Вес наблюдений по выручке согласует обучение с целевой метрикой.
5. Подбор Optuna добавляет небольшой прирост поверх.

По тепловой карте раздела 6.1 видно, где это работает: на агрегатах бустинг заметно точнее, а на
уровне отдельного товара простые модели не уступают.

## 8.3. Метрики качества (интерактивная таблица)

In [ ]:
#| label: plotly-metrics
#| code-fold: true
import plotly.graph_objects as go

st = summary.reset_index()
metrics_fig = go.Figure(go.Table(
    columnwidth=[2.4, 1, 1, 1, 1],
    header=dict(values=["модель", "WRMSSE", "RMSSE", "MASE", "Bias"],
                fill_color="#2c3e50", font=dict(color="white", size=13), align="left", height=30),
    cells=dict(values=[st["модель"], st["WRMSSE"], st["RMSSE"], st["MASE"], st["Bias"]],
               align="left", height=26)))
metrics_fig.update_layout(title="Метрики качества по моделям (среднее по фолдам)",
                          margin=dict(t=46, b=6, l=6, r=6), height=260)
metrics_fig.show()

## 8.4. Интерактивная панель: факт и все прогнозы

Каскадные выпадающие списки выбирают уровень (штат, магазин, отдел, товар). На графике факт, прогнозы всех
моделей на тестовых неделях (включаются кликом по легенде) и реальный прогноз итоговой модели на
4 недели вперёд. Сбоку RMSSE и Bias всех моделей на выбранном срезе. Данные считаются в Python,
переключение и отрисовку делает встроенный JS с plotly.

In [ ]:
#| label: plotly-panel
#| code-fold: true
import json
from IPython.display import HTML, display

HKEYS = ["state_id", "store_id", "dept_id", "item_id"]
PANEL_TOP = 15
ORIGIN, TEST_W = folds[-1]["origin"], folds[-1]["test_w"]
META = df[["id"] + HKEYS].drop_duplicates()
id2item = META.set_index("id")["item_id"]
hist = df[(df[TCOL] > ORIGIN - pd.Timedelta(weeks=22)) & (df[TCOL] <= ORIGIN)]
test_act = df[df[TCOL].isin(TEST_W)]
train_full = df[df[TCOL] <= ORIGIN]
vol = train_full.groupby("id", observed=True)["units"].sum()
mp = {nm: p.merge(META, on="id") for nm, p in model_preds.items()}
real = real_fc.merge(META, on="id")
_wk = lambda idx: [d.strftime("%Y-%m-%d") for d in idx]
_r = lambda v: [round(float(x), 1) for x in v]


@cached(ignore=["hist", "test_act", "train_full", "real", "mp", "vol"])
def panel_payload(hist, test_act, train_full, real, mp, vol, sig):
    E = pd.Series(dtype="float64")

    def nodes(tbl, val, keys):
        return {(k if isinstance(k, tuple) else (k,)): s.groupby(TCOL, observed=True)[val].sum()
                for k, s in tbl.groupby(keys, observed=True)}

    def pack(a, h, r, tr, mser):
        sc = float((tr.diff() ** 2).mean()); ty, met = {}, {}
        for nm in mp:
            p = mser.get(nm, a * 0).reindex(a.index).fillna(0)
            ty[nm] = _r(p.values); mse = float(((a - p) ** 2).mean())
            met[nm] = {"rmsse": round((mse / sc) ** 0.5, 3) if sc > 0 else None,
                       "bias": round(float((p.sum() - a.sum()) / a.sum()), 3) if a.sum() > 0 else None}
        return {"series": {"hx": _wk(h.index), "hy": _r(h.values), "tx": _wk(a.index), "ta": _r(a.values),
                           "ty": ty, "fx": _wk(r.index), "fy": _r(r.values)}, "metrics": met}

    def assemble(keys, ids=None):
        flt = (lambda t: t[t["id"].isin(ids)]) if ids is not None else (lambda t: t)
        H = nodes(flt(hist), "units", keys); R = nodes(flt(real), "pred", keys)
        A = nodes(flt(test_act), "units", keys); TR = nodes(flt(train_full), "units", keys)
        M = {nm: nodes(flt(mp[nm]), "pred", keys) for nm in mp}
        return {n: pack(a, H.get(n, E), R.get(n, E), TR.get(n, E), {nm: M[nm].get(n, E) for nm in mp})
                for n, a in A.items()}

    top = set()
    for (sto, dp), gg in META.groupby(["store_id", "dept_id"], observed=True):
        top |= set(vol.reindex(gg["id"]).sort_values(ascending=False).head(PANEL_TOP).index)
    L1 = assemble(["state_id"]); L2 = assemble(["state_id", "store_id"])
    L3 = assemble(["state_id", "store_id", "dept_id"]); L4 = assemble(HKEYS, top)
    glob = pack(test_act.groupby(TCOL)["units"].sum(), hist.groupby(TCOL)["units"].sum(),
                real.groupby(TCOL)["pred"].sum(), train_full.groupby(TCOL)["units"].sum(),
                {nm: mp[nm].groupby(TCOL)["pred"].sum() for nm in mp})
    data = {}
    for st in sorted(META["state_id"].unique()):
        sm = META[META.state_id == st]; stores = {}
        for sto in sorted(sm["store_id"].unique()):
            dm0 = sm[sm.store_id == sto]; depts = {}
            for dp in sorted(dm0["dept_id"].unique()):
                dm = dm0[dm0.dept_id == dp]
                tids = vol.reindex(dm["id"]).sort_values(ascending=False).head(PANEL_TOP).index
                items = {str(id2item[i]): L4[(st, sto, dp, id2item[i])] for i in tids
                         if (st, sto, dp, id2item[i]) in L4}
                depts[str(dp)] = {"agg": L3[(st, sto, dp)], "items": items, "it_order": list(items)}
            stores[str(sto)] = {"agg": L2[(st, sto)], "depts": depts,
                                "dp_order": [str(d) for d in sorted(dm0["dept_id"].unique())]}
        data[str(st)] = {"agg": L1[(st,)], "stores": stores,
                         "st_order": [str(s) for s in sorted(sm["store_id"].unique())]}
    return {"states": list(data), "data": data, "global": glob,
            "models": list(mp), "cutoff": ORIGIN.strftime("%Y-%m-%d")}


payload = panel_payload(hist, test_act, train_full, real, mp, vol, f"{DATA_SIG}-panel")
_data = json.dumps(payload, ensure_ascii=False)

import plotly.io as pio

# пустая фигура нужна, чтобы вывод ячейки сам встроил библиотеку plotly.js и контейнер графика:
# в песочнице вывода VSCode внешний CDN не грузится, поэтому plotly кладём прямо в вывод
_fig0 = go.Figure()
_fig0.update_layout(template="plotly_white", height=440, yaxis_title="units", xaxis_title="неделя",
                    margin={"l": 55, "r": 12, "t": 8, "b": 70})
_plot = pio.to_html(_fig0, include_plotlyjs=True, full_html=False, div_id="pn-plot",
                    config={"displayModeBar": False, "responsive": True})

_HTML = r"""
<style>
  .pn{background:#fff;padding:14px;border:1px solid #e3e3e3;border-radius:8px;margin:10px 0;font-family:-apple-system,'Segoe UI',sans-serif;color:#222;}
  .pn-c{display:flex;gap:12px;margin-bottom:10px;flex-wrap:wrap;align-items:flex-end;}
  .pn-c label{font-size:11px;font-weight:600;color:#666;display:block;margin-bottom:3px;}
  .pn-c select{padding:6px 9px;border:1px solid #bbb;border-radius:4px;font-size:13px;min-width:150px;background:#fafafa;}
  .pn-h{font-size:13px;font-weight:600;margin:4px 0 8px;color:#444;}
  .pn-w{display:flex;gap:12px;align-items:flex-start;flex-wrap:wrap;}
  .pn-plot-host{flex:1;min-width:480px;}
  .pn-k{width:300px;font-size:12px;}
  .pn-k table{border-collapse:collapse;width:100%;}
  .pn-k th{background:#37474f;color:#fff;padding:5px 8px;font-weight:600;}
  .pn-k td{padding:4px 8px;text-align:center;border-bottom:1px solid #eee;}
</style>
<div class="pn">
  <div class="pn-c">
    <div><label>Штат</label><select id="pn-s"></select></div>
    <div><label>Магазин</label><select id="pn-st"></select></div>
    <div><label>Отдел</label><select id="pn-d"></select></div>
    <div><label>Товар</label><select id="pn-it"></select></div>
  </div>
  <div class="pn-h" id="pn-title">-</div>
  <div class="pn-w"><div class="pn-plot-host">__PLOT__</div><div class="pn-k" id="pn-folds"></div></div>
</div>
<script>
(function(){
  function boot(){
    if(!window.Plotly || !document.getElementById("pn-plot")){return setTimeout(boot, 30);}
    var P=__DATA__, ALL="(все)", D=P.data, MODELS=P.models, CUT=P.cutoff;
    var PAL=["#7f8c8d","#2980b9","#16a085","#8e44ad","#e67e22","#27ae60"], COL={};
    MODELS.forEach(function(m,i){COL[m]=PAL[i%PAL.length];});
    var st={s:ALL,st:ALL,d:ALL,it:ALL}, $=function(id){return document.getElementById(id);};
    var eS=$("pn-s"),eSt=$("pn-st"),eD=$("pn-d"),eIt=$("pn-it"),eT=$("pn-title"),eF=$("pn-folds");
    var opt=function(s,a,v){s.innerHTML="";a.forEach(function(t){var o=document.createElement("option");o.value=t;o.textContent=t;s.appendChild(o);});s.value=v;};
    function node(){
      if(st.s===ALL)return P.global; var a=D[st.s];
      if(st.st===ALL)return a.agg; var b=a.stores[st.st];
      if(st.d===ALL)return b.agg; var c=b.depts[st.d];
      if(st.it===ALL)return c.agg; return c.items[st.it];
    }
    var f3=function(v){return v==null?"-":v.toFixed(3);}, sg=function(v){return v==null?"-":(v>=0?"+":"")+v.toFixed(3);};
    function draw(){
      try{
        var nd=node(); if(!nd){eT.textContent="нет данных для выбранного среза";return;}
        var s=nd.series, tr=[];
        tr.push({x:s.hx.concat(s.tx),y:s.hy.concat(s.ta),name:"факт",mode:"lines+markers",line:{color:"#111",width:2.4}});
        MODELS.forEach(function(m){tr.push({x:s.tx,y:(s.ty[m]||[]),name:m,mode:"lines+markers",line:{color:COL[m],width:1.6,dash:"dot"}});});
        tr.push({x:s.fx,y:s.fy,name:"реальный прогноз вперёд",mode:"lines+markers",line:{color:"#c0392b",width:2.6}});
        Plotly.react("pn-plot",tr,{template:"plotly_white",height:440,yaxis:{title:"units"},xaxis:{title:"неделя"},
          margin:{l:55,r:12,t:8,b:70},hovermode:"x unified",legend:{orientation:"h",y:-0.28},
          shapes:[{type:"line",x0:CUT,x1:CUT,yref:"paper",y0:0,y1:1,line:{color:"grey",width:1,dash:"dash"}}]},
          {responsive:true,displayModeBar:false});
        var lvl=st.it!==ALL?"товар":(st.d!==ALL?"отдел":(st.st!==ALL?"магазин":(st.s!==ALL?"штат":"итог")));
        eT.textContent=st.s+" / "+st.st+" / "+st.d+" / "+st.it+"  -  "+lvl+", последний фолд";
        var h="<b>RMSSE и Bias на срезе</b><table><tr><th>модель</th><th>RMSSE</th><th>Bias</th></tr>";
        MODELS.forEach(function(m){var r=(nd.metrics||{})[m]||{};h+="<tr><td style='text-align:left'>"+m+"</td><td>"+f3(r.rmsse)+"</td><td>"+sg(r.bias)+"</td></tr>";});
        eF.innerHTML=h+"</table>";
      }catch(e){eT.textContent="ошибка отрисовки: "+(e&&e.message?e.message:e); if(window.console)console.error(e);}
    }
    var fSt=function(){opt(eSt, st.s===ALL?[ALL]:[ALL].concat(D[st.s].st_order), st.st);};
    var fD=function(){opt(eD, (st.s!==ALL&&st.st!==ALL)?[ALL].concat(D[st.s].stores[st.st].dp_order):[ALL], st.d);};
    var fIt=function(){opt(eIt, (st.s!==ALL&&st.st!==ALL&&st.d!==ALL)?[ALL].concat(D[st.s].stores[st.st].depts[st.d].it_order||[]):[ALL], st.it);};
    opt(eS,[ALL].concat(P.states),ALL); fSt(); fD(); fIt();
    eS.onchange=function(){st.s=eS.value;st.st=ALL;st.d=ALL;st.it=ALL;fSt();fD();fIt();draw();};
    eSt.onchange=function(){st.st=eSt.value;st.d=ALL;st.it=ALL;fD();fIt();draw();};
    eD.onchange=function(){st.d=eD.value;st.it=ALL;fIt();draw();};
    eIt.onchange=function(){st.it=eIt.value;draw();};
    draw();
  }
  boot();
})();
</script>
"""
display(HTML(_HTML.replace("__DATA__", _data).replace("__PLOT__", _plot)))